# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_15 — Greeks: Pathwise Derivatives vs Likelihood Ratio

### Research question

How can Monte Carlo estimators compute option Greeks efficiently, and when should we use finite differences, pathwise derivatives, or likelihood-ratio estimators?

This notebook extends:

```text
02_06_monte_carlo_gbm_and_fat_tails
02_07_exotic_options_monte_carlo
02_08_dynamic_delta_hedging_simulation
```

Those notebooks priced options and simulated hedging. This notebook focuses on **sensitivities**:

$$
\Delta,\quad \Gamma,\quad \text{Vega}
$$
It covers:

1. finite-difference Greeks with common random numbers;
2. pathwise derivative estimators;
3. likelihood-ratio / score-function estimators;
4. variance and standard-error comparison;
5. vanilla call Delta and Vega;
6. digital option Delta as a discontinuous-payoff case;
7. estimator failure modes;
8. antithetic variates;
9. confidence intervals;
10. practical estimator selection rules;
11. saved estimator comparison reports and manifest.

The key idea:

> A Monte Carlo Greek is not just a derivative. It is a statistical estimator whose bias, variance, and validity depend on payoff smoothness and model parameterisation.

## 1. Three approaches to Monte Carlo Greeks

Suppose an option price is:

$$
V(\theta)=\mathbb E[g(X_\theta)]
$$
where $\theta$ is a parameter such as spot $S_0$ or volatility $\sigma$.

### 1.1 Finite difference estimator

$$
\frac{V(\theta+h)-V(\theta-h)}{2h}
$$
Advantages:

- simple;
- works for many black-box pricers;
- easy to implement.

Disadvantages:

- biased for finite $h$;
- can have high variance;
- bump-size selection is delicate;
- expensive if many Greeks are needed.

### 1.2 Pathwise derivative estimator

If the payoff is smooth enough:

$$
\begin{aligned}
\frac{\partial}{\partial\theta}\mathbb E[g(X_\theta)] &= \mathbb E\left[ g'(X_\theta)\frac{\partial X_\theta}{\partial\theta} \right]
\end{aligned}
$$
Advantages:

- often low variance;
- efficient for smooth payoffs;
- works naturally with automatic differentiation.

Disadvantages:

- can fail for discontinuous payoffs;
- may not directly handle indicators or barriers.

### 1.3 Likelihood-ratio estimator

If the density $f_\theta(x)$ is differentiable:

$$
\begin{aligned}
\frac{\partial}{\partial\theta} \mathbb E[g(X)] &= \mathbb E\left[ g(X) \frac{\partial}{\partial\theta}\log f_\theta(X) \right]
\end{aligned}
$$
The term:

$$
\frac{\partial}{\partial\theta}\log f_\theta(X)
$$
is called the **score**.

Advantages:

- works for discontinuous payoffs;
- useful for digital/barrier-style payoffs.

Disadvantages:

- can have high variance;
- requires density/score calculations;
- parameter-specific derivations are needed.

## 2. GBM terminal distribution

Under risk-neutral GBM:

$$
\begin{aligned}
S_T &= S_0 \exp \left[ (r-q-\frac{1}{2}\sigma^2)T+\sigma\sqrt{T}Z \right]
\end{aligned}
$$
where:

$$
Z\sim\mathcal N(0,1)
$$
For common-random-number finite differences, we keep the same $Z$ and perturb $S_0$ or $\sigma$.

For pathwise derivatives:

$$
\begin{aligned}
\frac{\partial S_T}{\partial S_0} &= \frac{S_T}{S_0}
\end{aligned}
$$
$$
\begin{aligned}
\frac{\partial S_T}{\partial \sigma} &= S_T \left( -\sigma T+\sqrt{T}Z \right)
\end{aligned}
$$

## 3. Likelihood-ratio scores under GBM

Let:

$$
Y=\log(S_T/S_0)
$$
Then:

$$
\begin{aligned}
Y &\sim \mathcal N \left( (r-q-\frac12\sigma^2)T, \sigma^2T \right)
\end{aligned}
$$
The score for $S_0$, using the terminal lognormal density, is:

$$
\begin{aligned}
\frac{\partial}{\partial S_0}\log f(S_T;S_0) &= \frac{Z}{S_0\sigma\sqrt T}
\end{aligned}
$$
The score for $\sigma$ is:

$$
\begin{aligned}
\frac{\partial}{\partial \sigma}\log f(S_T;\sigma) &= \frac{Z^2-1}{\sigma} \\
&\quad - \sqrt T Z
\end{aligned}
$$
So likelihood-ratio estimators are:

$$
\begin{aligned}
\Delta_{LR} &= e^{-rT}g(S_T) \frac{Z}{S_0\sigma\sqrt T}
\end{aligned}
$$
$$
\begin{aligned}
\text{Vega}_{LR} &= e^{-rT}g(S_T) \Bigg[ \frac{Z^2-1}{\sigma} \\
&\quad - \sqrt T Z \Bigg]
\end{aligned}
$$

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class GreeksMCConfig:
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    volatility: float = 0.20
    n_paths: int = 500_000
    seed: int = 42


config = GreeksMCConfig()
config

## 5. Analytical Black-Scholes references

We use analytical BSM Greeks as ground truth for vanilla and digital options.

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)


def bsm_d1_d2(s0, K, T, r, q, sigma):
    d1 = (log(s0 / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return d1, d2


def bsm_call_price(s0, K, T, r, q, sigma):
    d1, d2 = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(s0 * exp(-q*T) * normal_cdf(d1) - K * exp(-r*T) * normal_cdf(d2))


def bsm_call_delta(s0, K, T, r, q, sigma):
    d1, _ = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(exp(-q*T) * normal_cdf(d1))


def bsm_call_gamma(s0, K, T, r, q, sigma):
    d1, _ = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(exp(-q*T) * normal_pdf(d1) / (s0 * sigma * sqrt(T)))


def bsm_call_vega(s0, K, T, r, q, sigma):
    d1, _ = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(s0 * exp(-q*T) * normal_pdf(d1) * sqrt(T))


def bsm_digital_call_price(s0, K, T, r, q, sigma, cash_payoff=1.0):
    _, d2 = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(cash_payoff * exp(-r*T) * normal_cdf(d2))


def bsm_digital_call_delta(s0, K, T, r, q, sigma, cash_payoff=1.0):
    _, d2 = bsm_d1_d2(s0, K, T, r, q, sigma)
    return float(cash_payoff * exp(-r*T) * normal_pdf(d2) / (s0 * sigma * sqrt(T)))


reference_values = pd.Series({
    "call_price": bsm_call_price(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility),
    "call_delta": bsm_call_delta(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility),
    "call_gamma": bsm_call_gamma(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility),
    "call_vega": bsm_call_vega(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility),
    "digital_call_price": bsm_digital_call_price(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility),
    "digital_call_delta": bsm_digital_call_delta(config.s0, config.strike, config.maturity, config.risk_free_rate, config.dividend_yield, config.volatility)
})

reference_values

## 6. Simulate terminal GBM paths

We store both:

- the normal shock $Z$;
- terminal price $S_T$.

This lets every estimator use the same random numbers.

In [ ]:
def simulate_terminal_gbm(config: GreeksMCConfig, antithetic: bool = False):
    rng = np.random.default_rng(config.seed)

    if antithetic:
        half = config.n_paths // 2
        z_half = rng.standard_normal(half)
        z = np.concatenate([z_half, -z_half])
        if len(z) < config.n_paths:
            z = np.concatenate([z, rng.standard_normal(1)])
    else:
        z = rng.standard_normal(config.n_paths)

    drift = (
        config.risk_free_rate
        - config.dividend_yield
        - 0.5 * config.volatility**2
    ) * config.maturity

    diffusion = config.volatility * sqrt(config.maturity) * z
    st = config.s0 * np.exp(drift + diffusion)

    return pd.DataFrame({
        "z": z,
        "terminal_price": st
    })


sim = simulate_terminal_gbm(config, antithetic=False)

sim.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(np.log(sim["terminal_price"] / config.s0), bins=100, density=True)
plt.title("Simulated Terminal Log Returns")
plt.xlabel("log(S_T/S0)")
plt.ylabel("Density")
plt.show()

## 7. Estimator summary helper

Every estimator produces a pathwise sample.

We summarise:

- estimate;
- standard error;
- confidence interval;
- bias versus analytical reference;
- RMSE proxy from bias and variance.

In [ ]:
def estimator_summary(samples, label, reference):
    samples = np.asarray(samples, dtype=float)
    estimate = float(np.mean(samples))
    se = float(np.std(samples, ddof=1) / np.sqrt(len(samples)))
    bias = estimate - reference
    rmse_proxy = float(np.sqrt(bias**2 + se**2))

    return {
        "estimator": label,
        "n_paths": len(samples),
        "estimate": estimate,
        "standard_error": se,
        "lower_95": estimate - 1.96 * se,
        "upper_95": estimate + 1.96 * se,
        "reference": reference,
        "bias": bias,
        "abs_bias": abs(bias),
        "rmse_proxy": rmse_proxy
    }


def discounted_call_payoff(st, K, r, T):
    return exp(-r*T) * np.maximum(st - K, 0.0)


def discounted_digital_call_payoff(st, K, r, T, cash_payoff=1.0):
    return exp(-r*T) * cash_payoff * (st > K).astype(float)

## 8. Vanilla call Delta estimators

### Pathwise Delta

For a call:

$$
g(S_T)=(S_T-K)^+
$$
The pathwise Delta estimator is:

$$
\begin{aligned}
\Delta_{PW} &= e^{-rT} \mathbf 1_{\{S_T>K\}} \frac{S_T}{S_0}
\end{aligned}
$$
### Likelihood-ratio Delta

$$
\begin{aligned}
\Delta_{LR} &= e^{-rT}(S_T-K)^+ \frac{Z}{S_0\sigma\sqrt T}
\end{aligned}
$$
### Finite difference Delta

Using common random numbers:

$$
\begin{aligned}
\Delta_{FD} &= \frac{V(S_0+h)-V(S_0-h)}{2h}
\end{aligned}
$$

In [ ]:
def call_delta_pathwise(sim, config):
    st = sim["terminal_price"].to_numpy()
    return exp(-config.risk_free_rate * config.maturity) * (st > config.strike) * (st / config.s0)


def call_delta_likelihood_ratio(sim, config):
    st = sim["terminal_price"].to_numpy()
    z = sim["z"].to_numpy()
    payoff = np.maximum(st - config.strike, 0.0)
    score = z / (config.s0 * config.volatility * sqrt(config.maturity))
    return exp(-config.risk_free_rate * config.maturity) * payoff * score


def call_delta_finite_difference_crn(sim, config, bump):
    z = sim["z"].to_numpy()

    def st_from_s0(s0):
        drift = (
            config.risk_free_rate
            - config.dividend_yield
            - 0.5 * config.volatility**2
        ) * config.maturity
        diffusion = config.volatility * sqrt(config.maturity) * z
        return s0 * np.exp(drift + diffusion)

    st_up = st_from_s0(config.s0 + bump)
    st_down = st_from_s0(config.s0 - bump)

    price_up_samples = discounted_call_payoff(st_up, config.strike, config.risk_free_rate, config.maturity)
    price_down_samples = discounted_call_payoff(st_down, config.strike, config.risk_free_rate, config.maturity)

    return (price_up_samples - price_down_samples) / (2 * bump)


delta_samples = {
    "pathwise_delta": call_delta_pathwise(sim, config),
    "likelihood_ratio_delta": call_delta_likelihood_ratio(sim, config),
    "finite_difference_delta_crn_h0.5": call_delta_finite_difference_crn(sim, config, bump=0.5),
    "finite_difference_delta_crn_h0.1": call_delta_finite_difference_crn(sim, config, bump=0.1)
}

delta_report = pd.DataFrame([
    estimator_summary(samples, name, reference_values["call_delta"])
    for name, samples in delta_samples.items()
])

delta_report

In [ ]:
plt.figure(figsize=(10, 6))
for name, samples in delta_samples.items():
    clipped = np.clip(samples, np.quantile(samples, 0.005), np.quantile(samples, 0.995))
    plt.hist(clipped, bins=80, alpha=0.45, density=True, label=name)
plt.title("Vanilla Call Delta Estimator Sample Distributions")
plt.xlabel("Estimator sample value, clipped at 0.5%/99.5%")
plt.ylabel("Density")
plt.legend()
plt.show()

## 9. Vanilla call Vega estimators

### Pathwise Vega

$$
\begin{aligned}
\frac{\partial S_T}{\partial \sigma} &= S_T(-\sigma T+\sqrt{T}Z)
\end{aligned}
$$
So:

$$
\begin{aligned}
\text{Vega}_{PW} &= e^{-rT} \mathbf 1_{\{S_T>K\}} S_T(-\sigma T+\sqrt{T}Z)
\end{aligned}
$$
### Likelihood-ratio Vega

$$
\begin{aligned}
\text{Vega}_{LR} &= e^{-rT}(S_T-K)^+ \Bigg[ \frac{Z^2-1}{\sigma} \\
&\quad - \sqrt T Z \Bigg]
\end{aligned}
$$
### Finite difference Vega

$$
\begin{aligned}
\text{Vega}_{FD} &= \frac{V(\sigma+h)-V(\sigma-h)}{2h}
\end{aligned}
$$

In [ ]:
def call_vega_pathwise(sim, config):
    st = sim["terminal_price"].to_numpy()
    z = sim["z"].to_numpy()
    dst_dsigma = st * (-config.volatility * config.maturity + sqrt(config.maturity) * z)
    return exp(-config.risk_free_rate * config.maturity) * (st > config.strike) * dst_dsigma


def call_vega_likelihood_ratio(sim, config):
    st = sim["terminal_price"].to_numpy()
    z = sim["z"].to_numpy()
    payoff = np.maximum(st - config.strike, 0.0)
    score = ((z**2 - 1.0) / config.volatility) - sqrt(config.maturity) * z
    return exp(-config.risk_free_rate * config.maturity) * payoff * score


def call_vega_finite_difference_crn(sim, config, bump):
    z = sim["z"].to_numpy()

    def st_from_sigma(sigma):
        drift = (
            config.risk_free_rate
            - config.dividend_yield
            - 0.5 * sigma**2
        ) * config.maturity
        diffusion = sigma * sqrt(config.maturity) * z
        return config.s0 * np.exp(drift + diffusion)

    st_up = st_from_sigma(config.volatility + bump)
    st_down = st_from_sigma(config.volatility - bump)

    price_up_samples = discounted_call_payoff(st_up, config.strike, config.risk_free_rate, config.maturity)
    price_down_samples = discounted_call_payoff(st_down, config.strike, config.risk_free_rate, config.maturity)

    return (price_up_samples - price_down_samples) / (2 * bump)


vega_samples = {
    "pathwise_vega": call_vega_pathwise(sim, config),
    "likelihood_ratio_vega": call_vega_likelihood_ratio(sim, config),
    "finite_difference_vega_crn_h0.01": call_vega_finite_difference_crn(sim, config, bump=0.01),
    "finite_difference_vega_crn_h0.005": call_vega_finite_difference_crn(sim, config, bump=0.005)
}

vega_report = pd.DataFrame([
    estimator_summary(samples, name, reference_values["call_vega"])
    for name, samples in vega_samples.items()
])

vega_report

In [ ]:
plt.figure(figsize=(10, 6))
for name, samples in vega_samples.items():
    clipped = np.clip(samples, np.quantile(samples, 0.005), np.quantile(samples, 0.995))
    plt.hist(clipped, bins=80, alpha=0.45, density=True, label=name)
plt.title("Vanilla Call Vega Estimator Sample Distributions")
plt.xlabel("Estimator sample value, clipped at 0.5%/99.5%")
plt.ylabel("Density")
plt.legend()
plt.show()

## 10. Finite difference bump-size experiment

Finite difference estimators have a bias-variance tradeoff.

If bump $h$ is too large:

- truncation bias increases.

If bump $h$ is too small:

- numerical cancellation and Monte Carlo noise can dominate.

Using common random numbers helps a lot, but bump-size selection still matters.

In [ ]:
def bump_size_experiment(sim, config, delta_bumps, vega_bumps):
    rows = []

    for h in delta_bumps:
        samples = call_delta_finite_difference_crn(sim, config, bump=h)
        rows.append({
            "greek": "delta",
            "bump": h,
            **estimator_summary(samples, f"fd_delta_h{h}", reference_values["call_delta"])
        })

    for h in vega_bumps:
        samples = call_vega_finite_difference_crn(sim, config, bump=h)
        rows.append({
            "greek": "vega",
            "bump": h,
            **estimator_summary(samples, f"fd_vega_h{h}", reference_values["call_vega"])
        })

    return pd.DataFrame(rows)


bump_experiment = bump_size_experiment(
    sim,
    config,
    delta_bumps=[2.0, 1.0, 0.5, 0.25, 0.10, 0.05],
    vega_bumps=[0.05, 0.02, 0.01, 0.005, 0.0025]
)

bump_experiment[["greek", "bump", "estimate", "standard_error", "bias", "rmse_proxy"]]

In [ ]:
plt.figure(figsize=(10, 6))

for greek, group in bump_experiment.groupby("greek"):
    plt.plot(group["bump"], group["rmse_proxy"], marker="o", label=greek)

plt.xscale("log")
plt.yscale("log")
plt.title("Finite Difference RMSE Proxy by Bump Size")
plt.xlabel("Bump size")
plt.ylabel("RMSE proxy")
plt.legend()
plt.show()

## 11. Digital call: where pathwise fails

A cash-or-nothing digital call payoff is:

$$
g(S_T)=\mathbf 1_{\{S_T>K\}}
$$
The payoff is discontinuous.

The pathwise derivative of the indicator is zero almost everywhere, which would incorrectly suggest:

$$
\Delta_{PW}\approx0
$$
But the true Delta is not zero:

$$
\begin{aligned}
\Delta_{\text{digital}} &= e^{-rT} \frac{\phi(d_2)} {S_0\sigma\sqrt T}
\end{aligned}
$$
This is a classic case where likelihood-ratio methods are useful.

In [ ]:
def digital_delta_pathwise_wrong(sim, config):
    return np.zeros(len(sim))


def digital_delta_likelihood_ratio(sim, config):
    st = sim["terminal_price"].to_numpy()
    z = sim["z"].to_numpy()
    payoff_indicator = (st > config.strike).astype(float)
    score = z / (config.s0 * config.volatility * sqrt(config.maturity))
    return exp(-config.risk_free_rate * config.maturity) * payoff_indicator * score


def digital_delta_finite_difference_crn(sim, config, bump):
    z = sim["z"].to_numpy()

    def st_from_s0(s0):
        drift = (
            config.risk_free_rate
            - config.dividend_yield
            - 0.5 * config.volatility**2
        ) * config.maturity
        diffusion = config.volatility * sqrt(config.maturity) * z
        return s0 * np.exp(drift + diffusion)

    st_up = st_from_s0(config.s0 + bump)
    st_down = st_from_s0(config.s0 - bump)

    price_up_samples = discounted_digital_call_payoff(st_up, config.strike, config.risk_free_rate, config.maturity)
    price_down_samples = discounted_digital_call_payoff(st_down, config.strike, config.risk_free_rate, config.maturity)

    return (price_up_samples - price_down_samples) / (2 * bump)


digital_delta_samples = {
    "pathwise_delta_wrong_zero": digital_delta_pathwise_wrong(sim, config),
    "likelihood_ratio_delta": digital_delta_likelihood_ratio(sim, config),
    "finite_difference_delta_crn_h0.5": digital_delta_finite_difference_crn(sim, config, bump=0.5),
    "finite_difference_delta_crn_h0.1": digital_delta_finite_difference_crn(sim, config, bump=0.1)
}

digital_delta_report = pd.DataFrame([
    estimator_summary(samples, name, reference_values["digital_call_delta"])
    for name, samples in digital_delta_samples.items()
])

digital_delta_report

In [ ]:
plt.figure(figsize=(10, 6))
for name, samples in digital_delta_samples.items():
    clipped = np.clip(samples, np.quantile(samples, 0.005), np.quantile(samples, 0.995))
    plt.hist(clipped, bins=80, alpha=0.45, density=True, label=name)
plt.title("Digital Call Delta Estimator Sample Distributions")
plt.xlabel("Estimator sample value, clipped")
plt.ylabel("Density")
plt.legend()
plt.show()

## 12. Antithetic variates experiment

Antithetic variates can reduce variance.

We compare the same estimators using paired shocks:

$$
Z,\quad -Z
$$
This is not guaranteed to improve every estimator equally, but it is often helpful.

In [ ]:
config_antithetic = GreeksMCConfig(**{**asdict(config), "seed": 123})
sim_antithetic = simulate_terminal_gbm(config_antithetic, antithetic=True)

antithetic_reports = []

for name, fn, ref in [
    ("call_delta_pathwise", call_delta_pathwise, reference_values["call_delta"]),
    ("call_delta_likelihood_ratio", call_delta_likelihood_ratio, reference_values["call_delta"]),
    ("call_vega_pathwise", call_vega_pathwise, reference_values["call_vega"]),
    ("call_vega_likelihood_ratio", call_vega_likelihood_ratio, reference_values["call_vega"]),
    ("digital_delta_likelihood_ratio", digital_delta_likelihood_ratio, reference_values["digital_call_delta"]),
]:
    plain_samples = fn(sim, config)
    anti_samples = fn(sim_antithetic, config_antithetic)

    plain = estimator_summary(plain_samples, name + "_plain", ref)
    anti = estimator_summary(anti_samples, name + "_antithetic", ref)

    antithetic_reports.append({
        "greek_estimator": name,
        "plain_standard_error": plain["standard_error"],
        "antithetic_standard_error": anti["standard_error"],
        "variance_reduction_ratio": (plain["standard_error"] ** 2) / (anti["standard_error"] ** 2),
        "plain_estimate": plain["estimate"],
        "antithetic_estimate": anti["estimate"],
        "reference": ref
    })

antithetic_report = pd.DataFrame(antithetic_reports)

antithetic_report

## 13. Estimator ranking

We combine the reports into a single table.

The best estimator depends on the payoff and Greek:

- vanilla call Delta: pathwise is usually strong;
- vanilla call Vega: pathwise often performs well;
- digital Delta: pathwise fails, likelihood-ratio works;
- finite difference is robust but bump-dependent.

In [ ]:
combined_report = pd.concat([
    delta_report.assign(greek="vanilla_call_delta"),
    vega_report.assign(greek="vanilla_call_vega"),
    digital_delta_report.assign(greek="digital_call_delta")
], ignore_index=True)

combined_report_sorted = combined_report.sort_values(["greek", "rmse_proxy"])

combined_report_sorted[[
    "greek",
    "estimator",
    "estimate",
    "reference",
    "bias",
    "standard_error",
    "rmse_proxy"
]]

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = combined_report_sorted.copy()
plot_df["label"] = plot_df["greek"] + "\n" + plot_df["estimator"]
plt.barh(plot_df["label"], plot_df["rmse_proxy"])
plt.title("Greek Estimator RMSE Proxy Comparison")
plt.xlabel("RMSE proxy")
plt.ylabel("Estimator")
plt.show()

## 14. Practical estimator selection rules

| Payoff / Greek | Finite difference | Pathwise | Likelihood ratio |
|---|---:|---:|---:|
| Smooth vanilla Delta | good but bump-sensitive | usually excellent | valid but often higher variance |
| Smooth vanilla Vega | good but bump-sensitive | often strong | valid but may be noisy |
| Digital Delta | works with common random numbers but bump-sensitive | fails | useful |
| Barrier Greeks | tricky | often fails without smoothing | useful but can be high variance |
| Large portfolio Greeks | expensive | efficient with AD/AAD | useful for discontinuities |

Practical rule:

> Use pathwise when the payoff is smooth enough. Use likelihood-ratio when the payoff has discontinuities. Use finite differences as a benchmark and fallback.

## 15. Saving outputs

The notebook saves:

1. BSM analytical references;
2. vanilla call Delta estimator report;
3. vanilla call Vega estimator report;
4. finite-difference bump experiment;
5. digital Delta estimator report;
6. antithetic variance-reduction report;
7. combined ranking;
8. manifest.

In [ ]:
output_dir = Path("data/processed/greeks_pathwise_vs_likelihood_ratio_v1")
output_dir.mkdir(parents=True, exist_ok=True)

reference_path = output_dir / "analytical_references.csv"
delta_report_path = output_dir / "vanilla_call_delta_estimators.csv"
vega_report_path = output_dir / "vanilla_call_vega_estimators.csv"
bump_path = output_dir / "finite_difference_bump_experiment.csv"
digital_path = output_dir / "digital_call_delta_estimators.csv"
antithetic_path = output_dir / "antithetic_variance_reduction.csv"
combined_path = output_dir / "combined_estimator_ranking.csv"
manifest_path = output_dir / "manifest.json"

reference_values.to_frame("value").to_csv(reference_path)
delta_report.to_csv(delta_report_path, index=False)
vega_report.to_csv(vega_report_path, index=False)
bump_experiment.to_csv(bump_path, index=False)
digital_delta_report.to_csv(digital_path, index=False)
antithetic_report.to_csv(antithetic_path, index=False)
combined_report_sorted.to_csv(combined_path, index=False)

manifest = {
    "dataset_name": "greeks_pathwise_vs_likelihood_ratio_outputs",
    "schema_version": "greeks_pathwise_vs_likelihood_ratio_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "analytical_references": reference_values.to_dict(),
    "notes": [
        "Terminal GBM paths are simulated exactly.",
        "Finite difference estimators use common random numbers.",
        "Pathwise estimators are implemented for vanilla call Delta and Vega.",
        "Likelihood-ratio estimators use lognormal score functions.",
        "Digital call demonstrates discontinuous-payoff failure of the naive pathwise estimator.",
        "Antithetic variates are compared for selected estimators."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

reference_path, delta_report_path, vega_report_path, digital_path, manifest_path

## 16. Limitations

### 16.1 GBM terminal model only

This notebook uses exact terminal GBM.

Path-dependent Greeks require differentiating the whole path or using adjoint methods.

### 16.2 Simple payoff set

The notebook focuses on vanilla and digital calls.

Asian, barrier, lookback, callable, and multi-asset products introduce additional complications.

### 16.3 Gamma not fully treated

Gamma is easy analytically but more delicate in Monte Carlo.

Second derivatives often need mixed estimators, smoothing, finite differences, or adjoint techniques.

### 16.4 Likelihood-ratio variance

Likelihood-ratio estimators can have high variance, especially for deep OTM options or long maturities.

### 16.5 Finite difference bump choice

Finite differences depend on bump size.

Common random numbers reduce variance but do not eliminate truncation bias.

### 16.6 Discontinuity handling

The digital example shows pathwise failure.

For more complex discontinuities, conditional Monte Carlo, smoothing, or Malliavin methods may be needed.

## 17. Research frontier and extensions

Important extensions include:

1. **Adjoint Algorithmic Differentiation**  
   Efficiently compute many Greeks for large portfolios.

2. **Mixed estimators**  
   Combine pathwise and likelihood-ratio methods to reduce variance.

3. **Conditional Monte Carlo**  
   Smooth discontinuous payoffs before differentiating.

4. **Malliavin calculus Greeks**  
   Generalised integration-by-parts techniques.

5. **Automatic differentiation Monte Carlo**  
   Use JAX/PyTorch/autograd to differentiate simulations.

6. **Greeks under stochastic volatility**  
   Heston and local-stochastic volatility Greeks require more careful pathwise dynamics.

7. **Barrier and digital Greeks**  
   Discontinuous payoffs need smoothing, bridge conditioning, or LRM methods.

8. **GPU Greeks engine**  
   Portfolio Greeks can be batched and accelerated on GPU.

## 18. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_08_dynamic_delta_hedging_simulation**  
   Greeks in hedging PnL.

2. **02_13_multilevel_monte_carlo_pricing**  
   MLMC Greeks and variance reduction.

3. **02_16_adjoint_algorithmic_differentiation_intro**  
   AAD for large portfolios.

4. **06_12_gpu_monte_carlo_engine**  
   Batched price and Greek simulation.

5. **07_02_volatility_surface_pricing_and_alpha**  
   Greeks for surface-based option portfolios.

## 19. Summary

This notebook compared Monte Carlo Greek estimators.

It showed how to:

1. compute analytical BSM Greek references;
2. simulate terminal GBM paths;
3. estimate Delta by finite differences;
4. estimate Delta by pathwise derivatives;
5. estimate Delta by likelihood ratios;
6. estimate Vega by finite differences;
7. estimate Vega by pathwise derivatives;
8. estimate Vega by likelihood ratios;
9. demonstrate pathwise failure for digital Delta;
10. compare variance and standard errors;
11. test antithetic variance reduction;
12. save estimator diagnostics.

The key computational takeaway:

> Greek estimation is estimator design. Correctness and variance matter as much as the formula.

The key financial takeaway:

> Smooth payoffs favour pathwise estimators; discontinuous payoffs often require likelihood-ratio, smoothing, or conditional methods.

## 20. Further reading

- Glasserman, *Monte Carlo Methods in Financial Engineering*.
- Broadie and Glasserman on estimating Greeks using simulation.
- Giles, "Smoking Adjoints: fast evaluation of Greeks in Monte Carlo calculations."
- Capriotti, "Likelihood Ratio Method and Algorithmic Differentiation."
- Literature on Malliavin Greeks, conditional Monte Carlo, and adjoint algorithmic differentiation.